# Healthcare Operations Intelligence Platform

## Notebook 4: Machine Learning for Operational Risk Prediction

### Objective

This notebook develops a predictive model to identify hospital operational congestion risk.

The model aims to support proactive decision-making by predicting whether operational pressure is likely to occur.

Step 2: Import Libraries

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import roc_auc_score

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import joblib

Step 3: Load KPI Dataset

In [2]:
DATA_PATH = Path(
    "../data/processed/analytics_features_v3.csv"
)

df = pd.read_csv(DATA_PATH)

df.head()

,date,department,patient_volume,avg_wait_time,avg_treatment_time,occupancy_rate,staff_available,staff_ratio,risk_score,current_congestion_flag,congestion_flag
0,2026-01-01,Emergency,152,71,122,90.0,14,10.857143,78.947913,1,1
1,2026-01-02,Emergency,199,72,31,90.0,14,14.214286,84.070829,1,0
2,2026-01-03,Emergency,107,41,137,90.0,14,7.642857,62.308628,0,1
3,2026-01-04,Emergency,239,81,76,90.0,14,17.071429,91.723257,1,1
4,2026-01-05,Emergency,122,58,47,90.0,14,8.714286,70.683339,1,1


Step 4: Check Target Distribution

In [3]:
df["congestion_flag"].value_counts()

congestion_flag
0    455
1    261
Name: count, dtype: int64

Step 5: Define Features and Target

In [4]:
X = df.drop(
    columns=[
        "congestion_flag",
        "current_congestion_flag",
        "risk_score",
        "department",
        "date"
    ],
    errors="ignore"
)

y = df["congestion_flag"]

Step 6: Train/Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Model 1: Logistic Regression

In [6]:
log_model = LogisticRegression(
    max_iter=1000
)


log_model.fit(
    X_train,
    y_train
)


log_predictions = log_model.predict(
    X_test
)

Evaluate Baseline

In [7]:
print(
    classification_report(
        y_test,
        log_predictions
    )
)

              precision    recall  f1-score   support

           0       0.75      0.89      0.81        92
           1       0.71      0.46      0.56        52

    accuracy                           0.74       144
   macro avg       0.73      0.68      0.69       144
weighted avg       0.73      0.74      0.72       144



Model 2: Random Forest

In [8]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)


rf_model.fit(
    X_train,
    y_train
)


rf_predictions = rf_model.predict(
    X_test
)

Evaluate Random Forest

In [9]:
accuracy = accuracy_score(
    y_test,
    rf_predictions
)

precision = precision_score(
    y_test,
    rf_predictions,
    zero_division=0
)

recall = recall_score(
    y_test,
    rf_predictions,
    zero_division=0
)

f1 = f1_score(
    y_test,
    rf_predictions,
    zero_division=0
)


print("Accuracy:", accuracy)

print("Precision:", precision)

print("Recall:", recall)

print("F1 Score:", f1)

Accuracy: 0.75
Precision: 0.6379310344827587
Recall: 0.7115384615384616
F1 Score: 0.6727272727272727


In [10]:
probabilities = rf_model.predict_proba(
    X_test
)[:,1]


auc = roc_auc_score(
    y_test,
    probabilities
)


print(
    "ROC-AUC:",
    auc
)

ROC-AUC: 0.8124999999999999


Step 8: Confusion Matrix

In [11]:
cm = confusion_matrix(
    y_test,
    rf_predictions
)

cm

array([[71, 21],
       [15, 37]])

This tells us:

correctly identified risks
missed risks
false alarms

Step 9: Feature Importance

In [12]:
importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance":
    rf_model.feature_importances_

})


importance.sort_values(
    "Importance",
    ascending=False
)

,Feature,Importance
3,occupancy_rate,0.328199
4,staff_available,0.232311
5,staff_ratio,0.154109
0,patient_volume,0.102810
2,avg_treatment_time,0.092648
1,avg_wait_time,0.089925


Step 10: Save Model

In [13]:
Path("models").mkdir(
        exist_ok=True
)

joblib.dump(
    rf_model,
    "models/congestion_prediction_model.pkl"
)

['models/congestion_prediction_model.pkl']

# Machine Learning Findings

The predictive model was developed to identify operational congestion risk.

Two algorithms were evaluated:

1. Logistic Regression as a baseline model.

2. Random Forest as the final predictive model.

The Random Forest model was selected because it captures complex relationships between operational variables.

Feature importance analysis identifies the main operational factors contributing to congestion risk.

The final model will be integrated into the Streamlit dashboard to provide real-time operational predictions.